In [26]:
import pandas as pd

df = pd.read_csv("btc_1h.csv", index_col=0, parse_dates=True)# ucitavanje podataka
df.head()

,Open,High,Low,Close,Volume
Datetime,,,,,
2017-08-17 04:00:00,4261.48,4313.62,4261.32,4308.83,47.181009
2017-08-17 05:00:00,4308.83,4328.69,4291.37,4315.32,23.234916
2017-08-17 06:00:00,4330.29,4345.45,4309.37,4324.35,7.229691
2017-08-17 07:00:00,4316.62,4349.99,4287.41,4349.99,4.443249
2017-08-17 08:00:00,4333.32,4377.85,4333.32,4360.69,0.972807


In [27]:
#!pip install ta
import ta

#return
df["return_1h"] = df["Close"].pct_change()# koliko se cena promenila u % zato što model lakše uči promene nego apsolutne cene.
df['daily_return'] = df['Close'].pct_change(24)# Dnevni prinos
#mean/std
df["rolling_mean_24h"] = df["Close"].rolling(24).mean()# prosečna cena zadnja 24h
df["rolling_std_24h"] = df["Close"].rolling(24).std()# volatilnost
#lag
df["close_lag_6h"] = df["Close"].shift(6)# cena pre 6h
df["close_lag_12h"] = df["Close"].shift(12)# cena pre 12h
df["close_lag_24h"] = df["Close"].shift(24)# cena pre 24h
df['close_lag_48h'] = df['Close'].shift(48)  # Cena pre 48h
df['close_lag_168h'] = df['Close'].shift(168)  # Cena pre 7 dana

# RSI
df['RSI_14'] = ta.momentum.RSIIndicator(df['Close'], window=14).rsi()

# MACD
macd = ta.trend.MACD(df['Close'], window_slow=26, window_fast=12, window_sign=9)
df['MACD'] = macd.macd()
df['MACD_signal'] = macd.macd_signal()  # MACD signal linija
df['MACD_hist'] = macd.macd_diff()  # MACD histogram

# Bollinger Bands
bb = ta.volatility.BollingerBands(df['Close'], window=20, window_dev=2)
df['BB_mavg'] = bb.bollinger_mavg()  # Srednja Bollinger linija
df['BB_upper'] = bb.bollinger_hband()  # Gornja Bollinger linija
df['BB_lower'] = bb.bollinger_lband()  # Donja Bollinger linija

# SMA i EMA
df['SMA_20'] = ta.trend.SMAIndicator(df['Close'], window=20).sma_indicator()
df['EMA_20'] = ta.trend.EMAIndicator(df['Close'], window=20).ema_indicator()

# ATR (Average True Range)
df['ATR_14'] = ta.volatility.AverageTrueRange(high=df['High'], low=df['Low'], close=df['Close'], window=14).average_true_range()

# ROC (Rate of Change) - Procenat promene cene u poslednjem periodu
df['ROC'] = ta.momentum.ROCIndicator(df['Close'], window=12).roc()

df = df.dropna()# brisanje jer rolling i lag nekada stvaraju prazne vrednosti.
df.head() #ispis :)

,Open,High,Low,Close,Volume,return_1h,daily_return,rolling_mean_24h,rolling_std_24h,close_lag_6h,...,MACD,MACD_signal,MACD_hist,BB_mavg,BB_upper,BB_lower,SMA_20,EMA_20,ATR_14,ROC
Datetime,,,,,,,,,,,,,,,,,,,,,
2017-08-24 04:00:00,4113.58,4148.19,4090.39,4113.98,32.247571,0.000097,0.007440,4145.985833,52.439849,4114.20,...,14.272129,26.070185,-11.798056,4158.5800,4249.760962,4067.399038,4158.5800,4123.466923,73.916662,-2.281456
2017-08-24 05:00:00,4113.98,4177.64,4113.49,4132.09,28.158769,0.004402,0.019469,4149.273750,48.709115,4114.01,...,13.525285,23.561205,-10.035920,4155.0285,4244.510883,4065.546117,4155.0285,4124.288168,73.219043,-0.820398
2017-08-24 06:00:00,4132.09,4177.18,4131.91,4133.42,29.921536,0.000322,0.013093,4151.499583,46.579934,4131.00,...,12.892113,21.427387,-8.535274,4150.2985,4233.637800,4066.959200,4150.2985,4125.157866,71.222683,-0.240143
2017-08-24 07:00:00,4153.97,4173.99,4133.41,4153.32,32.851584,0.004814,0.019250,4154.767917,43.628501,4140.91,...,13.836584,19.909226,-6.072642,4146.0650,4219.123982,4073.006018,4146.0650,4127.839974,69.033920,0.880481
2017-08-24 08:00:00,4153.80,4206.88,4153.32,4200.00,32.275428,0.011239,0.018429,4157.934583,44.054251,4131.92,...,18.142633,19.555907,-1.413274,4145.2810,4215.620914,4074.941086,4145.2810,4134.712358,67.928640,1.981352


In [28]:
#dodaje se future return
df["future_close_24h"] = df["Close"].shift(-24)# uzimamo cenu 24h u budućnosti
df["future_return_24h"] = (df["future_close_24h"] - df["Close"]) / df["Close"]# Model ne predviđa cenu nego promenu

def classify_direction(x):
    if x > 0.02:
        return 2      # UP
    elif x < -0.02:
        return 0      # DOWN
    else:
        return 1      # STABLE

#izbacuje NaN
df["direction_24h"] = df["future_return_24h"].apply(classify_direction)

df = df.dropna()
df["direction_24h"].value_counts(normalize=True)

direction_24h
1    0.592607
2    0.217604
0    0.189789
Name: proportion, dtype: float64

In [29]:
# -----------------------------
# 1) Temporal split
# -----------------------------

target_col = "direction_24h"

features = [
    "Open","High","Low","Close","Volume",
    "return_1h","daily_return",
    "rolling_mean_24h","rolling_std_24h",
    "close_lag_6h","close_lag_12h","close_lag_24h","close_lag_48h","close_lag_168h",
    "RSI_14","MACD","MACD_signal","MACD_hist",
    "ATR_14","ROC",
    "SMA_20","EMA_20","BB_upper","BB_lower"
]

# split: 60% train, 20% val, 20% test
n = len(df)
train_end = int(n * 0.6)
val_end = int(n * 0.8)

train_df = df.iloc[:train_end]
val_df   = df.iloc[train_end:val_end]
test_df  = df.iloc[val_end:]

print("Train / Val / Test sizes:", len(train_df), len(val_df), len(test_df))


Train / Val / Test sizes: 44718 14906 14906


In [30]:
# -----------------------------
# 2) Priprema X i y
# -----------------------------


X_train = train_df[features].values
y_train = train_df[target_col].values

X_val = val_df[features].values
y_val = val_df[target_col].values

X_test = test_df[features].values
y_test = test_df[target_col].values

In [31]:
# -----------------------------
# 3) Definisanje i treniranje XGBClassifier
# -----------------------------

from xgboost import XGBClassifier


xgb_clf = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    eval_metric='mlogloss'
)

xgb_clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [32]:
# -----------------------------
# 2) Predikcija
# -----------------------------

y_pred = xgb_clf.predict(X_test)

In [33]:
# -----------------------------
# 3) Evaluacija
# -----------------------------

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import numpy as np


accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')  # weighted zbog nebalansiranih klasa
report = classification_report(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print("\nXGBoost Classification Evaluation on Test Set")
print(f"Accuracy: {accuracy:.4f} | F1-Score: {f1:.4f}")
print("\nClassification Report:\n", report)
print("Confusion Matrix:\n", conf_matrix)

# Feature importance
importances = xgb_clf.feature_importances_
print("\nFeature Importance:")
for f, imp in zip(features, importances):
    print(f"{f}: {imp:.4f}")


XGBoost Classification Evaluation on Test Set
Accuracy: 0.2333 | F1-Score: 0.2197

Classification Report:
               precision    recall  f1-score   support

           0       0.15      0.74      0.25      2366
           1       0.56      0.16      0.25     10126
           2       0.21      0.02      0.04      2414

    accuracy                           0.23     14906
   macro avg       0.31      0.31      0.18     14906
weighted avg       0.44      0.23      0.22     14906

Confusion Matrix:
 [[1751  550   65]
 [8308 1670  148]
 [1604  753   57]]

Feature Importance:
Open: 0.0308
High: 0.0429
Low: 0.0473
Close: 0.0563
Volume: 0.0377
return_1h: 0.0158
daily_return: 0.0277
rolling_mean_24h: 0.0546
rolling_std_24h: 0.0342
close_lag_6h: 0.0348
close_lag_12h: 0.0319
close_lag_24h: 0.0477
close_lag_48h: 0.0513
close_lag_168h: 0.0500
RSI_14: 0.0263
MACD: 0.0326
MACD_signal: 0.0394
MACD_hist: 0.0241
ATR_14: 0.0675
ROC: 0.0205
SMA_20: 0.0471
EMA_20: 0.0661
BB_upper: 0.0566
BB_lower: 0